<a href="https://colab.research.google.com/github/sumbuz28/2014/blob/master/policy_acts_and_ordinance_analysis__using_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install langchain langchain-community langchain-ollama langchain-core chromadb pypdf

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_ollama import OllamaLLM
import os

/tmp/ipykernel_8563/809723842.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [3]:
from google.colab import files

uploaded = files.upload()


Saving Disability Services Policy.pdf to Disability Services Policy (1).pdf
Saving Code of Conduct (1).pdf to Code of Conduct (1) (1).pdf
Saving HIT Student Registration Policy.pdf to HIT Student Registration Policy (1).pdf
Saving SEXUAL HARRASSMENT POLICY.pdf to SEXUAL HARRASSMENT POLICY (1).pdf
Saving Admissions Policy.pdf to Admissions Policy (1).pdf
Saving HIT Study Policy amended 2023.pdf to HIT Study Policy amended 2023 (1).pdf
Saving School Governance Ordinance 2005.pdf to School Governance Ordinance 2005 (1).pdf
Saving ORDINANCE 15.pdf to ORDINANCE 15 (1).pdf
Saving ACADEMIC STAFF PROMOTION ORDINANCE 23.pdf to ACADEMIC STAFF PROMOTION ORDINANCE 23 (1).pdf
Saving Ordinance-15.pdf to Ordinance-15 (1).pdf
Saving HIT-BOARD-CHARTER-FINAL.pdf to HIT-BOARD-CHARTER-FINAL (1).pdf
Saving CODE-OF-ETHICS-FOR-BOARD-MEMBERS.pdf to CODE-OF-ETHICS-FOR-BOARD-MEMBERS (1).pdf
Saving HIT-CODE-OF-ETHICS-FOR-STAFF.pdf to HIT-CODE-OF-ETHICS-FOR-STAFF (1).pdf
Saving Code of Conduct.pdf to Code of Cond

In [4]:

import glob

pdf_files = glob.glob("/content/*.pdf")
all_documents = []

for file_path in pdf_files:
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    all_documents.extend(docs)




In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# 3. Split the loaded PDF documents into chunks
chunks = text_splitter.split_documents(all_documents)


In [6]:
print(f"Total loaded documents: {len(all_documents)}")
print(f"Total loaded chunked_documents: {len(chunks)}")

Total loaded documents: 416
Total loaded chunked_documents: 940


In [7]:
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-l6-v2")
vector_store = Chroma.from_documents(chunks,embeddings)
retriever = vector_store.as_retriever(search_kwargs = {"k":3})

/tmp/ipykernel_8563/53627930.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-l6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama server in the background using Python subprocess
import subprocess
import time

print("Starting Ollama server...")
# Subprocess lets it run in the background without blocking your notebook cells
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a few seconds to initialize and bind to port 11434
time.sleep(5)

# 3. Verify that the server is up and responding
!curl http://localhost:11434
!ollama pull llama3.2

Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Cleaning up old version at /usr/local/lib/o

In [9]:
model = OllamaLLM(model='llama3.2')

In [ ]:
template =  """
you are an expert in analysing policy ordinances and acts of universities
here are some policies and acts:{policy_and_acts}
and the question:{question}
    """

prompt =ChatPromptTemplate.from_template(template)
chain = prompt|model

while True:
  question = input("what is your question (q to quit): ")
  print("--------------------------------------------------------")
  if question == 'q':
    break

  policy_and_acts = retriever.invoke(question)
  result = chain.invoke({"policy_and_acts":policy_and_acts,"question":question})
  print(result)

what is your question (q to quit): in 150 words summarize the code of conduct at this university
--------------------------------------------------------
After analyzing the provided policy ordinances, I can summarize the Code of Conduct for this university as follows:

The Code of Conduct is a comprehensive policy document that outlines the expected behavior and standards of conduct for all students, faculty, and staff members. It emphasizes the importance of maintaining a respectful and inclusive learning environment, free from harassment, bullying, and discrimination.

The document highlights the following key principles:

* Respect for diversity, equity, and inclusion
* Zero tolerance for harassment, bullying, and discrimination
* Adherence to university policies and procedures
* Confidentiality and protection of sensitive information
* Responsibility for own actions and consequences
* Commitment to academic integrity and honesty

The Code of Conduct also outlines the procedures fo